# Scenario: File Processors API

**QE Perspective:** The `file_processors` API was missing from the OGX distro config, breaking file_search / vector store file ingestion. We validate that:

- The `file_processors` provider is registered (pypdf or auto).
- **Direct upload** processing returns chunks with content and metadata.
- **file_id-based** processing works for files already uploaded via `/v1/files`.
- Chunk metadata references the source file correctly.

Uses the OpenAI client throughout: typed `files.create`/`files.delete` for `/v1/files`, and a `/v1alpha`-scoped sub-client (`client.with_options(base_url=…)`) for the `file_processors` endpoint, which has no typed SDK method. Uploads pass `Content-Type: multipart/form-data` so the SDK sends them as multipart (it fills in the boundary).

## Setup

Load base URL from environment; create httpx client. MODEL is not needed for file processing.

In [ ]:
import os
from io import BytesIO
from openai import OpenAI

base_url = os.environ.get("BASE_URL")
assert base_url, "BASE_URL must be set"

host = base_url.rstrip("/").removesuffix("/v1")
client = OpenAI(api_key="no-key-needed", base_url=f"{host}/v1")

# file_processors is a v1alpha endpoint with no typed SDK method, so we use a
# client scoped to /v1alpha. Content-Type: multipart/form-data tells the SDK to
# send the upload as multipart (it fills in the boundary).
fp_client = client.with_options(base_url=f"{host}/v1alpha")
FORM = {"headers": {"Content-Type": "multipart/form-data"}}
print(f"OGX: {host}/v1")

## Verify file_processors provider is registered

The fix for RHAIENG-5114 adds a `file_processors` provider to the distro config. Verify it exists.

In [ ]:
providers_resp = client.get("/providers", cast_to=object)
providers = providers_resp.get("data", [])
fp_providers = [p for p in providers if p.get("api") == "file_processors"]

assert len(fp_providers) > 0, (
    "No file_processors provider registered — RHAIENG-5114 regression: "
    "file_processors API is missing from distro config"
)

for p in fp_providers:
    print(f"  {p['provider_id']} ({p['provider_type']})")

## Test 1: Direct file upload processing

Upload a text file directly to `/v1alpha/file-processors/process` and verify chunks are returned.

In [ ]:
test_content = b"Hello from file_processors notebook test"

body = fp_client.post(
    "/file-processors/process",
    files={"file": ("test.txt", test_content, "text/plain")},
    options=FORM,
    cast_to=object,
)
assert "chunks" in body, "Response missing 'chunks' field"
assert isinstance(body["chunks"], list), "chunks should be a list"
assert len(body["chunks"]) >= 1, "Expected at least one chunk"

chunk = body["chunks"][0]
assert chunk.get("content") and isinstance(chunk["content"], str), (
    "Chunk missing non-empty string 'content'"
)
assert "metadata" in body and "processor" in body["metadata"], (
    "Response missing metadata.processor"
)

print(f"Chunks: {len(body['chunks'])}")
print(f"Content: {chunk['content']!r}")
print(f"Processor: {body['metadata']['processor']}")

## Test 2: Process file by file_id

Upload a file via `/v1/files`, then process it by referencing the `file_id`. This is the path used by vector store file ingestion (the flow broken by RHAIENG-5114).

In [ ]:
buf = BytesIO(test_content)
buf.name = "test-fp.txt"
uploaded = client.files.create(file=buf, purpose="assistants")
file_id = uploaded.id
assert file_id, "File upload did not return an ID"
print(f"Uploaded file_id: {file_id}")

try:
    body = fp_client.post(
        "/file-processors/process",
        files={"file_id": (None, file_id)},
        options=FORM,
        cast_to=object,
    )
    assert len(body["chunks"]) >= 1, "Expected at least one chunk"

    chunk = body["chunks"][0]
    assert chunk["content"], "Chunk content should not be empty"

    chunk_meta = chunk.get("metadata", {})
    assert chunk_meta.get("file_id") == file_id, (
        f"Chunk metadata file_id mismatch: expected {file_id}, got {chunk_meta.get('file_id')}"
    )

    print(f"Chunks: {len(body['chunks'])}")
    print(f"Content: {chunk['content']!r}")
    print(f"Chunk file_id: {chunk_meta.get('file_id')}")
finally:
    client.files.delete(file_id)
    print(f"Deleted file {file_id}")

## Test 3: Owner-encrypted PDF processing (RHAIENG-5857)

Owner-encrypted PDFs restrict printing/copying but require no password to read.
Upstream [ogx-ai/ogx#5670](https://github.com/ogx-ai/ogx/pull/5670) added a blanket `is_encrypted` check
that rejects these. This test catches that regression.

In [ ]:
from pathlib import Path
from openai import APIStatusError

# Cwd is tests/functional/ when notebooks run via pytest
pdf_path = Path("../fixtures/sample-encrypted.pdf")
assert pdf_path.exists(), f"Fixture not found: {pdf_path}"

with open(pdf_path, "rb") as f:
    try:
        data = fp_client.post(
            "/file-processors/process",
            files={"file": ("sample-encrypted.pdf", f, "application/pdf")},
            options=FORM,
            cast_to=object,
        )
    except APIStatusError as e:
        raise AssertionError(
            f"Server rejected owner-encrypted PDF (HTTP {e.status_code}). "
            f"Regression: blanket is_encrypted check — see ogx-ai/ogx#6181"
        )

chunks = data.get("chunks", [])
assert len(chunks) > 0, "No chunks returned from encrypted PDF"

marker = "OGX-RAG-SMOKE-7f3a9c2e8b1d4f06"
all_text = " ".join(c.get("content", "") for c in chunks)
assert marker in all_text, f"Expected marker '{marker}' not found in extracted text"

print(f"Owner-encrypted PDF: {len(chunks)} chunks, marker found")